# AIDS-2 Mini Project: Breast Cancer Classification Using Deep Learning

**Aim:** Analyse the supplied BreastMNIST dataset and train, evaluate, and compare four deep-learning architectures for binary image classification. This notebook follows the staged workflow of the reference notebook: dataset inspection, EDA, preprocessing, model training, validation-based comparison, held-out test evaluation, visualisations, and a measured conclusion.

> **Important:** This is an educational experiment on a small research dataset, not a clinically validated diagnostic tool.

## 1. Problem Statement

Build a binary image classifier using BreastMNIST ultrasound images. Compare four neural-network architectures using the same predefined training, validation, and test splits. Select the leading model using **validation macro F1-score**, then report test-set performance separately.

## 2. Dataset Description

The attached `breastmnist(1).npz` file contains six arrays: `train_images`, `val_images`, `test_images`, and their matching label arrays. Images are 28 × 28 grayscale pixels. The dataset already provides separate splits, so this notebook preserves them rather than randomly splitting the data again.

The two numeric labels are retained as provided by the dataset. For reporting, refer to them as **Class 0** and **Class 1** unless you verify the exact class mapping from the dataset's official documentation.

## 3. Import Required Libraries

This notebook uses TensorFlow/Keras for all four models, plus scikit-learn for evaluation. Google Colab normally includes these libraries.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 4. Configuration — Upload the Dataset

Upload the supplied `.npz` file to the Colab session if it is not already present. The cell below opens the upload dialog only when the configured file path is missing.

In [ ]:
DATA_PATH = "/content/breastmnist(1).npz"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No dataset file was uploaded.")
        DATA_PATH = "/content/" + next(iter(uploaded.keys()))
    except ImportError as exc:
        raise FileNotFoundError(
            "Dataset not found. Place breastmnist(1).npz beside this notebook or update DATA_PATH."
        ) from exc

print("Dataset path:", DATA_PATH)

## 5. Load the Dataset

The dataset archive contains the official train/validation/test arrays. We will check that every image has a matching label.

In [ ]:
with np.load(DATA_PATH) as data:
    required_keys = ["train_images", "val_images", "test_images",
                     "train_labels", "val_labels", "test_labels"]
    missing_keys = [key for key in required_keys if key not in data.files]
    if missing_keys:
        raise ValueError(f"Dataset is missing required arrays: {missing_keys}")
    X_train_raw = data["train_images"].copy()
    X_val_raw = data["val_images"].copy()
    X_test_raw = data["test_images"].copy()
    y_train = data["train_labels"].reshape(-1).astype("int32")
    y_val = data["val_labels"].reshape(-1).astype("int32")
    y_test = data["test_labels"].reshape(-1).astype("int32")

print("Train images:", X_train_raw.shape, "labels:", y_train.shape)
print("Validation images:", X_val_raw.shape, "labels:", y_val.shape)
print("Test images:", X_test_raw.shape, "labels:", y_test.shape)
print("Pixel range before normalisation:", X_train_raw.min(), "to", X_train_raw.max())
print("Unique labels:", np.unique(np.concatenate([y_train, y_val, y_test])))
assert len(X_train_raw) == len(y_train)
assert len(X_val_raw) == len(y_val)
assert len(X_test_raw) == len(y_test)
assert set(np.unique(np.concatenate([y_train, y_val, y_test]))) <= {0, 1}

### 5.1 Basic Dataset Description

In [ ]:
dataset_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test", "Total"],
    "Images": [len(y_train), len(y_val), len(y_test), len(y_train) + len(y_val) + len(y_test)],
    "Image height": [X_train_raw.shape[1], X_val_raw.shape[1], X_test_raw.shape[1], X_train_raw.shape[1]],
    "Image width": [X_train_raw.shape[2], X_val_raw.shape[2], X_test_raw.shape[2], X_train_raw.shape[2]],
})
display(dataset_summary)
print("Image dtype:", X_train_raw.dtype)
print("Train pixel min/max:", X_train_raw.min(), X_train_raw.max())

### 5.2 Check Labels and Class Balance

In [ ]:
class_balance = pd.DataFrame({
    "Train": pd.Series(y_train).value_counts().sort_index(),
    "Validation": pd.Series(y_val).value_counts().sort_index(),
    "Test": pd.Series(y_test).value_counts().sort_index(),
}).fillna(0).astype(int)
class_balance.index.name = "Class label"
display(class_balance)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, labels, split_name in zip(axes, [y_train, y_val, y_test], ["Train", "Validation", "Test"]):
    counts = pd.Series(labels).value_counts().reindex([0, 1], fill_value=0)
    ax.bar(["Class 0", "Class 1"], counts.values)
    ax.set_title(f"{split_name} class distribution")
    ax.set_ylabel("Number of images")
plt.tight_layout()
plt.show()

### 5.3 Visualise Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, range(10)):
    ax.imshow(X_train_raw[idx], cmap="gray")
    ax.set_title(f"Label: {y_train[idx]}")
    ax.axis("off")
plt.suptitle("Sample BreastMNIST Training Images", y=1.02)
plt.tight_layout()
plt.show()

## 6. Data Preprocessing

Convert pixel values to the 0–1 range and add the single-channel dimension required by CNNs. The ANN receives flattened images; the CNN models receive image-shaped tensors. The predefined splits are preserved.

In [ ]:
X_train = X_train_raw.astype("float32") / 255.0
X_val = X_val_raw.astype("float32") / 255.0
X_test = X_test_raw.astype("float32") / 255.0

X_train_cnn = X_train[..., np.newaxis]
X_val_cnn = X_val[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]

X_train_ann = X_train.reshape(len(X_train), -1)
X_val_ann = X_val.reshape(len(X_val), -1)
X_test_ann = X_test.reshape(len(X_test), -1)

print("ANN input:", X_train_ann.shape)
print("CNN input:", X_train_cnn.shape)
print("Normalised pixel range:", X_train.min(), "to", X_train.max())
print("Label counts (train):", dict(zip(*np.unique(y_train, return_counts=True))))

### 6.1 Calculate Class Weights

Class weights reduce the influence of class imbalance during training. They are calculated from the training labels only.

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = {int(label): float(weight) for label, weight in zip(classes, weights)}
print("Class weights:", class_weight_dict)

## 7. Training Configuration

Early stopping restores the weights from the epoch with the lowest validation loss. Reduce-on-plateau lowers the learning rate when validation loss stops improving. All models use the same training split and validation split.

In [ ]:
EPOCHS = 50
BATCH_SIZE = 32


def make_callbacks():
    return [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
        ),
    ]

trained_models = {}
training_histories = {}
validation_predictions = {}
validation_records = []
test_predictions = {}
test_records = []


def evaluate_predictions(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision (Macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (Macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def register_model_result(model_name, model, history, x_val, x_test):
    val_probs = model.predict(x_val, verbose=0).reshape(-1)
    val_pred = (val_probs >= 0.5).astype("int32")
    validation_predictions[model_name] = val_pred
    validation_records.append({"Model": model_name, **evaluate_predictions(y_val, val_pred)})
    trained_models[model_name] = model
    training_histories[model_name] = history

    test_probs = model.predict(x_test, verbose=0).reshape(-1)
    test_predictions[model_name] = (test_probs >= 0.5).astype("int32")


def compile_binary_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
    )
    return model

## 8. Model 1 — Artificial Neural Network (ANN)

The ANN uses flattened pixels as input and dense layers to learn image-level patterns. It is included as a fully connected deep-learning baseline.

In [ ]:
ann_model = models.Sequential([
    layers.Input(shape=(28 * 28,)),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.35),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(1, activation="sigmoid"),
], name="ANN")
compile_binary_model(ann_model)
ann_model.summary()
ann_history = ann_model.fit(
    X_train_ann, y_train,
    validation_data=(X_val_ann, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict, callbacks=make_callbacks(), verbose=1,
)
register_model_result("ANN", ann_model, ann_history, X_val_ann, X_test_ann)

## 9. Model 2 — LeNet-5-Style CNN

This compact CNN uses convolution, average pooling, and dense layers. It is suitable for small grayscale images. It is a LeNet-style implementation rather than a claim of an exact historical reproduction.

In [ ]:
lenet_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(6, kernel_size=5, activation="tanh", padding="same"),
    layers.AveragePooling2D(pool_size=2),
    layers.Conv2D(16, kernel_size=5, activation="tanh"),
    layers.AveragePooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(120, activation="tanh"),
    layers.Dense(84, activation="tanh"),
    layers.Dense(1, activation="sigmoid"),
], name="LeNet_Style_CNN")
compile_binary_model(lenet_model)
lenet_model.summary()
lenet_history = lenet_model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict, callbacks=make_callbacks(), verbose=1,
)
register_model_result("LeNet-5-style CNN", lenet_model, lenet_history, X_val_cnn, X_test_cnn)

## 10. Model 3 — Custom CNN

The custom CNN learns spatial features through convolution and max-pooling layers, followed by dense layers for binary classification.

In [ ]:
cnn_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Conv2D(96, 3, padding="same", activation="relu"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.35),
    layers.Dense(1, activation="sigmoid"),
], name="Custom_CNN")
compile_binary_model(cnn_model)
cnn_model.summary()
cnn_history = cnn_model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict, callbacks=make_callbacks(), verbose=1,
)
register_model_result("Custom CNN", cnn_model, cnn_history, X_val_cnn, X_test_cnn)

## 11. Model 4 — VGG-Style CNN

The VGG-style network stacks small 3 × 3 convolutions in blocks. This compact version is adjusted for 28 × 28 grayscale images and is trained from scratch.

In [ ]:
vgg_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Dropout(0.20),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(96, 3, padding="same", activation="relu"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(96, activation="relu"),
    layers.Dropout(0.40),
    layers.Dense(1, activation="sigmoid"),
], name="VGG_Style_CNN")
compile_binary_model(vgg_model)
vgg_model.summary()
vgg_history = vgg_model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict, callbacks=make_callbacks(), verbose=1,
)
register_model_result("VGG-style CNN", vgg_model, vgg_history, X_val_cnn, X_test_cnn)

## 12. Validation Results — Compare the Four Models

Use validation macro F1-score to rank the models. Macro F1 gives equal weight to both classes. The test set remains separate and is not used to choose the winning architecture.

In [ ]:
validation_results_df = pd.DataFrame(validation_records).sort_values(
    "Macro F1", ascending=False
).reset_index(drop=True)
print("Validation comparison (ranked by Macro F1):")
display(validation_results_df.round(4))
assert len(validation_results_df) == 4

### 12.1 Validation Macro F1 Comparison

In [ ]:
plot_df = validation_results_df.sort_values("Macro F1", ascending=True)
plt.figure(figsize=(9, 5))
bars = plt.barh(plot_df["Model"], plot_df["Macro F1"])
plt.xlim(0, 1)
plt.xlabel("Macro F1-score")
plt.title("Validation Macro F1 — Four Deep-Learning Models")
for bar in bars:
    plt.text(min(bar.get_width() + 0.01, 0.95), bar.get_y() + bar.get_height()/2,
             f"{bar.get_width():.3f}", va="center")
plt.tight_layout()
plt.show()

### 12.2 Compare Validation Metrics

In [ ]:
metric_cols = ["Accuracy", "Balanced Accuracy", "Precision (Macro)", "Recall (Macro)", "Macro F1"]
ax = validation_results_df.set_index("Model")[metric_cols].plot(
    kind="bar", figsize=(13, 5), ylim=(0, 1), width=0.8
)
ax.set_title("Validation Metrics Across Four Models")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 13. Final Hold-Out Test Evaluation

Evaluate all four trained models on the untouched test split. These values are the final reported test metrics; they should be copied from the actual run, not estimated in advance.

In [ ]:
test_records = []
for model_name, predictions in test_predictions.items():
    test_records.append({"Model": model_name, **evaluate_predictions(y_test, predictions)})

test_results_df = pd.DataFrame(test_records).sort_values("Macro F1", ascending=False).reset_index(drop=True)
print("Held-out test-set results:")
display(test_results_df.round(4))

### 13.1 Validation vs Test Results

In [ ]:
comparison_df = validation_results_df.merge(
    test_results_df, on="Model", how="left", suffixes=(" (Validation)", " (Test)")
)
display(comparison_df.round(4))
comparison_df.set_index("Model")[["Accuracy (Validation)", "Accuracy (Test)", "Macro F1 (Validation)", "Macro F1 (Test)"]].plot(
    kind="bar", figsize=(12, 5), ylim=(0, 1)
)
plt.title("Validation vs Test Performance")
plt.ylabel("Score")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 14. Classification Report and Confusion Matrices

The confusion matrix and class-wise precision/recall/F1-score help show which labels are predicted correctly and which are confused. Labels are shown as Class 0 and Class 1.

In [ ]:
best_model_name = validation_results_df.iloc[0]["Model"]
print("Best model selected by validation Macro F1:", best_model_name)
print("\nClassification report for the selected model on the test set:")
print(classification_report(y_test, test_predictions[best_model_name], labels=[0, 1],
                            target_names=["Class 0", "Class 1"], zero_division=0))

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, (name, predictions) in zip(axes.flat, test_predictions.items()):
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Class 0", "Class 1"]).plot(
        ax=ax, colorbar=False, values_format="d"
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 15. Neural-Network Training Curves

Training and validation loss/accuracy are plotted for all four models. A widening gap between training and validation curves may indicate overfitting.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 18))
for row, (name, history) in enumerate(training_histories.items()):
    axes[row, 0].plot(history.history["accuracy"], label="Training accuracy")
    axes[row, 0].plot(history.history["val_accuracy"], label="Validation accuracy")
    axes[row, 0].set_title(f"{name} — Accuracy")
    axes[row, 0].set_xlabel("Epoch")
    axes[row, 0].set_ylabel("Accuracy")
    axes[row, 0].legend()

    axes[row, 1].plot(history.history["loss"], label="Training loss")
    axes[row, 1].plot(history.history["val_loss"], label="Validation loss")
    axes[row, 1].set_title(f"{name} — Loss")
    axes[row, 1].set_xlabel("Epoch")
    axes[row, 1].set_ylabel("Loss")
    axes[row, 1].legend()
plt.tight_layout()
plt.show()

## 16. Save Trained Models and Results

Save the Keras models and CSV result tables to the Colab session. Download them if you want to keep them after the runtime ends.

In [ ]:
os.makedirs("saved_models", exist_ok=True)
for model_name, model in trained_models.items():
    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")
    model.save(f"saved_models/{safe_name}.keras")

validation_results_df.to_csv("validation_results.csv", index=False)
test_results_df.to_csv("test_results.csv", index=False)
print("Saved models:", os.listdir("saved_models"))
print("Saved validation_results.csv and test_results.csv")

## 17. Conclusion — Generated from Measured Results

Run this cell after all model-training and evaluation cells. The conclusion is based on measured validation/test scores rather than assuming the largest network performs best.

In [ ]:
best_val = validation_results_df.iloc[0]
best_test = test_results_df.loc[test_results_df["Model"] == best_val["Model"]].iloc[0]
print("FINAL PERFORMANCE SUMMARY")
print("=" * 60)
print("Dataset: BreastMNIST")
print("Training images:", len(y_train))
print("Validation images:", len(y_val))
print("Test images:", len(y_test))
print("Models compared:", ", ".join(validation_results_df["Model"].tolist()))
print("Best model by validation Macro F1:", best_val["Model"])
print(f"Validation Macro F1: {best_val['Macro F1']:.4f}")
print(f"Test Accuracy: {best_test['Accuracy']:.4f}")
print(f"Test Macro F1: {best_test['Macro F1']:.4f}")
print("\nConclusion: The models were compared using a fixed validation split and evaluated on a separate test split.")
print("These results are for an educational dataset and must not be interpreted as clinical diagnostic performance.")

## 18. References

1. MedMNIST / BreastMNIST dataset documentation: https://medmnist.com/
2. TensorFlow Keras documentation: https://www.tensorflow.org/guide/keras
3. scikit-learn metrics documentation: https://scikit-learn.org/stable/modules/model_evaluation.html

**Reproducibility note:** Results can vary with software versions, hardware, and stochastic training. Run all cells from top to bottom in a fresh Colab runtime for a complete execution.